**2.3.2**

In [ ]:
# Gini coefficient within each age group

# Create an empty list to store the Gini coefficient for each age
gini_by_age = []

# Compute the Gini coefficient separately for each age
for t, age in enumerate(model_2_1.ages):
    gini_by_age.append(gini(model_2_1.income[:, t]))

# Convert to an array for convenience
gini_by_age = np.array(gini_by_age)
ages = np.asarray(model_2_1.ages)

# Plot the within-age Gini coefficient over the life cycle
plt.plot(ages, gini_by_age, label='Within age group')

# Add the pooled Gini coefficient as a horizontal reference line
plt.axhline(gini_pooled, linestyle='--', color='black', label='All ages pooled')

# Add labels and title
plt.xlabel('Age')
plt.ylabel('Gini coefficient')
plt.title('Inequality over the life cycle')

# Add legend
plt.legend()

# Show the figure
plt.show()

# Print the Gini coefficient at selected ages
for age in [18, 25, 35, 45, 60, ages[-1]]:
    t = int(np.where(ages == age)[0][0])
    print(f"Age {age}:", round(gini_by_age[t], 4))

# Find the age where the within-age measure crosses the pooled measure
above_pooled = gini_by_age > gini_pooled

if above_pooled.any():
    print("Crossing age:", ages[np.argmax(above_pooled)])
else:
    print("The within-age Gini never exceeds the pooled Gini")

The Gini coefficient for the pooled sample is 0.378, and the Lorenz curve lies clearly
below the 45-degree line.

Within a given age group, inequality follows a pronounced life-cycle profile. It is
zero at age 18, where everybody is still in education and receives the same student
grant, and then rises almost monotonically from 0.196 at age 25 to 0.452 at age 65, as
the multiplicative shocks accumulate and unemployment spells depreciate human capital.

The comparison with the pooled measure is therefore not one-directional. Up to age 51
the within-age Gini coefficient is *below* the pooled one, because pooling adds a
between-age component: a 60-year-old employed worker and a 20-year-old student differ
in income for purely life-cycle reasons, and this variation is counted as inequality
when all ages are pooled. From age 52 onwards the within-age coefficient *exceeds* the
pooled one, because by then the dispersion accumulated within the cohort is larger than
the dispersion generated by pooling ages together.

The within-age measure is the more meaningful notion of inequality here, since it
compares individuals who are otherwise similar. We return to this distinction in
question 2.4.

## Question 2.4 - What drives inequality?

In [ ]:
# Check that the flexible module reproduces the baseline exactly

baseline = model_2_4.simulate()

print("identical income:", np.array_equal(baseline['income'], model_2_1.income))
print("identical employment:", np.array_equal(baseline['employed'], model_2_1.employed))

In [ ]:
# Switch off the components of the model one at a time

specifications = {
    'Baseline': {},
    'No educational differences': {'educational_differences': False},
    'No shocks to human capital': {'shocks': False},
    'No depreciation when unemployed': {'depreciation': False},
    'No unemployment': {'unemployment': False},
}

# Select a single age group for the within-age comparison
age_group = 45
t_group = int(np.where(np.asarray(model_2_1.ages) == age_group)[0][0])

# Create an empty dictionary to store the results
results = {}

# Run the alternative simulations
for name, switches in specifications.items():

    sim = model_2_4.simulate(**switches)

    results[name] = {
        'pooled': gini(sim['income']),
        'within': gini(sim['income'][:, t_group]),
        'mean': sim['income'].mean(),
    }

# Print the results as a table
print(f"{'Simulation':34s}{'Pooled':>9s}{'Age 45':>9s}{'Mean y':>9s}")

for name, r in results.items():
    print(f"{name:34s}{r['pooled']:9.4f}{r['within']:9.4f}{r['mean']:9.3f}")

In [ ]:
# Plot the change in the Gini coefficient relative to the baseline

# Names of the alternative simulations
names = [name for name in specifications if name != 'Baseline']

# Changes relative to the baseline
d_pooled = [results[n]['pooled'] - results['Baseline']['pooled'] for n in names]
d_within = [results[n]['within'] - results['Baseline']['within'] for n in names]

# Position of the bars
x = np.arange(len(names))
width = 0.35

# Plot the two sets of bars next to each other
plt.bar(x - width/2, d_pooled, width, label='All ages pooled')
plt.bar(x + width/2, d_within, width, label=f'Age {age_group}')

# Add a horizontal line at zero
plt.axhline(0, color='black', linewidth=0.8)

# Add labels and title
plt.xticks(x, names, rotation=30, ha='right')
plt.ylabel('Change in Gini coefficient')
plt.title('Change in inequality relative to the baseline')

# Add legend
plt.legend()

# Adjust spacing
plt.tight_layout()

# Show the figure
plt.show()

The shock to human capital dominates. Switching it off lowers the pooled Gini from
0.378 to 0.279 and the age-45 Gini from 0.340 to 0.214: $\psi$ is multiplicative and
never reversed, so the variance of log human capital grows with years worked.
Educational differences come second (-0.045 pooled, -0.048 within age); depreciation
and unemployment matter little.

The last two simulations show the case in the hint. Removing depreciation or
unemployment lowers the age-45 Gini (to 0.329 and 0.326) but raises the pooled one (to
0.383 and 0.382). Within an age group the sign is intuitive: unemployment creates
dispersion between otherwise identical workers. The pooled measure moves the other way
because it also contains a between-age component: without unemployment, average income
rises from 1.47 to 1.82 while students stay at the grant of 0.45, so the age gap widens
by more than the within-age dispersion falls. The pooled Gini therefore mixes
inequality between individuals with mechanical age differences.

One caveat: switching off educational differences requires choosing an education for
everybody. We assign the medium one. The pooled Gini is sensitive to this (0.298, 0.333
and 0.382 with short, medium and long), but the age-45 value stays between 0.284 and
0.300, so the conclusion holds.

## Question 2.5 - Extension: more risk

In [ ]:
# Simulate the model for different degrees of health risk

# Values of the annual probability of a health shock
p_health_values = [0.000, 0.005, 0.010, 0.020, 0.030]

# Create empty lists to store the results
ever_disabled = []
mean_income_health = []
gini_pooled_health = []
gini_age_health = []

# Index of age 60
t60 = int(np.where(np.asarray(model_2_1.ages) == 60)[0][0])

# Run one simulation for each value of p_health
for p in p_health_values:

    sim = model_2_4.simulate(p_health=p)

    # Share of individuals who are disabled at the end of the life cycle
    ever_disabled.append(sim['disabled'][:, -1].mean())

    mean_income_health.append(sim['income'].mean())
    gini_pooled_health.append(gini(sim['income']))
    gini_age_health.append(gini(sim['income'][:, t60]))

# Print the results as a table
print(f"{'p_health':>10s}{'Disabled':>10s}{'Mean y':>10s}{'Pooled':>10s}{'Age 60':>10s}")

for i, p in enumerate(p_health_values):
    print(f"{p:10.3f}{ever_disabled[i]:10.3f}{mean_income_health[i]:10.3f}"
          f"{gini_pooled_health[i]:10.4f}{gini_age_health[i]:10.4f}")

In [ ]:
# Gini coefficient over the life cycle with and without health risk

# Baseline and extended simulation
sim_base = model_2_4.simulate()
sim_health = model_2_4.simulate(p_health=0.010)

# Compute the within-age Gini coefficient in both simulations
gini_age_base = np.array([gini(sim_base['income'][:, t])
                          for t in range(len(model_2_1.ages))])
gini_age_ext = np.array([gini(sim_health['income'][:, t])
                         for t in range(len(model_2_1.ages))])

# Plot the two life-cycle profiles
plt.plot(model_2_1.ages, gini_age_base, label='Baseline')
plt.plot(model_2_1.ages, gini_age_ext, label='With health risk')

# Add labels and title
plt.xlabel('Age')
plt.ylabel('Gini coefficient')
plt.title('Within-age inequality with and without health risk')

# Add legend
plt.legend()

# Show the figure
plt.show()

In [ ]:
# Compare the income distribution at age 60 with and without health risk

# Use the same bins for both histograms
bins = np.linspace(0, np.percentile(sim_base['income'][:, t60], 99), 60)

# Plot the two distributions on top of each other
plt.hist(sim_base['income'][:, t60], bins=bins, alpha=0.6, label='Baseline')
plt.hist(sim_health['income'][:, t60], bins=bins, alpha=0.6, label='With health risk')

# Add labels and title
plt.xlabel('Income')
plt.ylabel('Number of individuals')
plt.title('Income distribution at age 60')

# Add legend
plt.legend()

# Show the figure
plt.show()

# Compare the disabled with the rest of the population at age 60
is_disabled = sim_health['disabled'][:, t60]
income_60 = sim_health['income'][:, t60]

print("Share disabled at age 60:", round(is_disabled.mean(), 3))
print("Mean income, disabled:", round(income_60[is_disabled].mean(), 3))
print("Mean income, others:", round(income_60[~is_disabled].mean(), 3))

In [ ]:
# Is it the amount of non-employment or its persistence that matters?

# Non-employment rate at age 60 in the simulation with health risk
in_labor_market = model_2_1.ages[t60] >= sim_health['entry_age']

target = (
    (in_labor_market & ~sim_health['employed'][:, t60]).sum()
    / in_labor_market.sum()
)

# A purely temporary risk with the same steady-state non-employment rate
# requires a job-separation probability of sigma = u*lambda/(1-u)
sigma_alt = target * model_2_4.job_finding / (1 - target)

sim_temporary = model_2_4.simulate(job_separation_alt=sigma_alt)

print("Non-employment rate at age 60:", round(target, 4))
print("Implied job-separation probability:", round(sigma_alt, 4))
print()

print(f"{'':22s}{'Pooled':>9s}{'Age 60':>9s}{'Mean y':>9s}")

print(f"{'Permanent risk':22s}{gini(sim_health['income']):9.4f}"
      f"{gini(sim_health['income'][:, t60]):9.4f}{sim_health['income'].mean():9.3f}")

print(f"{'Temporary risk':22s}{gini(sim_temporary['income']):9.4f}"
      f"{gini(sim_temporary['income'][:, t60]):9.4f}{sim_temporary['income'].mean():9.3f}")

Health risk raises within-age inequality throughout the life cycle. With
$p^{h} = 0.01$, 36.5 per cent of the cohort is hit by age 65 and the age-60 Gini rises
from 0.428 to 0.447. The effect is cumulative in age: at 25 it rises only from 0.196 to
0.211, at 45 from 0.340 to 0.361. The histogram shows why: the distribution becomes
bimodal, with the disabled frozen at 60 per cent of their last wage (mean 0.88 at age
60 against 2.00 for the rest) while everybody else keeps accumulating human capital.

The pooled Gini is non-monotonic: it rises slightly for small $p^{h}$ but falls below
the baseline for $p^{h} \geq 0.02$. This is the mechanism of 2.4 in reverse. Health
shocks cut average income (1.47 to 1.08 at $p^{h}=0.03$), pushing older workers down
towards the grant of 0.45 and compressing the between-age component even as within-age
inequality keeps rising.

Finally, persistence matters in itself, not just the amount of non-employment. Setting
$\sigma$ to match the age-60 non-employment rate gives a pooled Gini of 0.350 against
0.380 under permanent risk, though the within-age gap is small (0.434 against 0.447):
the effect works mainly through the lower tail. The comparison is limited, however,
since the temporary version destroys far more human capital (mean income 0.80 against
1.30), as $\sigma = 0.37$ leaves almost everybody in long depreciating spells.